# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent.

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev`
4. Install gymnasium with the needed extras

In [1]:
%pip install -q swig
%pip install -q gymnasium[box2d,classic_control]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 30.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## The Lunar Lander Environment

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

Gymnasim environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percetps and returns an action, we need write glue code that connects the environment with the agent function.

In [2]:
import gymnasium as gym

def run_episode(agent_function, max_steps=1000):
    """Run one episode in the LunarLander-v3 environment using the provided agent."""
    total_reward = 0
    # Initialize the environment
    env = gym.make("LunarLander-v3", render_mode="human")

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        env.render()

        if terminated:
            print(f"Final Reward: {reward}")
            break

    env.close()
    return total_reward

Note: `env.render()` shows the environment when the notebook is locally run (e.g., in VScode). On Colab, you cannot see the environment because the code is run on a headless server (i.e., a server without a display). There are some workarounds you can google.

## Example: A Random Agent

We ranomly return one of the actions. The environment accepts the integers 0-3.


In [ ]:
import numpy as np

def random_agent_function(observation):
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

print(f"Total reward = ", run_episode(random_agent_function))

Obs: [-0.00584946  1.4113935  -0.5925037   0.02102461  0.00678486  0.13421082
  0.          0.        ] -> Action: 3
Obs: [-0.01161881  1.4113     -0.5815905  -0.00418938  0.01138758  0.09206373
  0.          0.        ] -> Action: 0
Obs: [-0.01738844  1.4106067  -0.5816051  -0.03084769  0.01598763  0.09200925
  0.          0.        ] -> Action: 0
Obs: [-0.02315807  1.4093139  -0.5816186  -0.05752049  0.02058671  0.09199005
  0.          0.        ] -> Action: 1
Obs: [-0.02900619  1.4074309  -0.59143674 -0.08379414  0.02714769  0.13123177
  0.          0.        ] -> Action: 1
Obs: [-0.03491926  1.4049442  -0.59960276 -0.11068996  0.03534171  0.16389547
  0.          0.        ] -> Action: 0
Obs: [-0.04083281  1.4018584  -0.5996276  -0.1373632   0.04353412  0.16386344
  0.          0.        ] -> Action: 1
Obs: [-0.04683352  1.3981589  -0.6105747  -0.16476153  0.05392568  0.20785034
  0.          0.        ] -> Action: 3
Obs: [-0.05276747  1.3938673  -0.60218036 -0.19108091  0.0626207

## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [7]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


In [8]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.3:
        return Act.MAIN.value

    return Act.NO_OP.value

run_episode(rocket_agent_function)

Obs: [ 0.00169287  1.416169    0.17144838  0.233283   -0.00195478 -0.03883555
  0.          0.        ] -> Action: 0
Obs: [ 0.00338564  1.4208399   0.1712167   0.2075877  -0.00387374 -0.03838224
  0.          0.        ] -> Action: 0
Obs: [ 0.00507841  1.4249113   0.17122272  0.18095016 -0.00579163 -0.03836165
  0.          0.        ] -> Action: 0
Obs: [ 0.00677128  1.4283829   0.17122838  0.15427889 -0.00770929 -0.03835658
  0.          0.        ] -> Action: 0
Obs: [ 0.00846415  1.4312544   0.17123407  0.12761009 -0.00962665 -0.03835059
  0.          0.        ] -> Action: 0
Obs: [ 0.0101572   1.4335258   0.17123975  0.10094149 -0.01154369 -0.03834459
  0.          0.        ] -> Action: 0
Obs: [ 0.01185017  1.4351972   0.17124541  0.07427297 -0.01346043 -0.03833862
  0.          0.        ] -> Action: 0
Obs: [ 0.01354332  1.4362688   0.17125106  0.04760439 -0.01537688 -0.03833267
  0.          0.        ] -> Action: 0
Obs: [ 0.01523647  1.4367405   0.17125669  0.02093574 -0.0172930

np.float64(-336.67356724242325)

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [16]:
import numpy as np

def run_episode_test(agent_function, max_steps = 1000):
    """Run one episode in the LunarLander-v3 environment using the provided agent."""

    # Initialise the environment
    env = gym.make("LunarLander-v3", render_mode=None)

    # Reset the environment to generate the first observation
    observation, info = env.reset()

    # run one episode (max. 1000 steps)
    for _ in range(max_steps):
        # call the agent to select an action
        action = agent_function(observation)

        # step (transition) through the environment with the action
        observation, reward, terminated, truncated, info = env.step(action)

        if terminated:
            break

    env.close()
    return reward

def run_episodes(agent_function, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode_test(agent_function) for _ in range(n)]

rewards = run_episodes(rocket_agent_function)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100

This is not great!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

In [ ]:
# Code goes here

In [21]:
# [pos_x, pos_y, vel_x, vel_y, angle, angular_velocity, left_leg_contact, right_leg_contact]
def heuristic_agent_function(Obs):
  pos_x, pos_y, vel_x, vel_y, angle, angular_velocity, left_leg_contact, right_leg_contact = Obs
  if vel_y < -0.5:
      return 2

  if angle > 0.1 and angular_velocity >= -0.2:
      return 3
  elif angle < -0.1 and angular_velocity <= 0.2:
      return 1
  if pos_x > 0.3:
      return 1
  elif pos_x < -0.3:
      return 3

  if pos_y < 0.3:
      if abs(angle) > 0.05:
          return 3 if angle > 0 else 1
      if vel_y < -0.3:
          return 2
  return 0

In [22]:
run_episode(heuristic_agent_function, 100)

Obs: [ 0.0058589   1.4073106   0.59341305 -0.16044106 -0.00678206 -0.13441697
  0.          0.        ] -> Action: 0
Obs: [ 0.0117178   1.4031243   0.5926114  -0.186103   -0.01342423 -0.13285506
  0.          0.        ] -> Action: 0
Obs: [ 0.01757698  1.3983386   0.59263206 -0.21277931 -0.02006238 -0.13277526
  0.          0.        ] -> Action: 0
Obs: [ 0.02343636  1.3929533   0.5926524  -0.23945388 -0.02669964 -0.1327576
  0.          0.        ] -> Action: 0
Obs: [ 0.02929592  1.3869684   0.59267175 -0.26612556 -0.03333585 -0.13273662
  0.          0.        ] -> Action: 0
Obs: [ 0.03515577  1.3803842   0.59269136 -0.2927969  -0.03997099 -0.13271537
  0.          0.        ] -> Action: 0
Obs: [ 0.04101581  1.3732004   0.59271085 -0.3194684  -0.0466051  -0.13269429
  0.          0.        ] -> Action: 0
Obs: [ 0.04687605  1.3654174   0.59273016 -0.3461401  -0.05323814 -0.1326732
  0.          0.        ] -> Action: 0
Obs: [ 0.05273657  1.3570348   0.5927495  -0.37281194 -0.05987014 

np.float64(44.285218246623856)

In [31]:
rewards = run_episodes(heuristic_agent_function, 1000)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 100, -100, -100, -100, -100, -100, 100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,